# Matched KGW and SWEET tuning

Run this notebook on a **Colab GPU through VS Code**. Upload `.tmp/codeguard-tuning-colab.zip` first.

The default pilot tests nine configurations on 64 HumanEval tasks: **640 new completions** (64 shared clean and 576 watermarked), plus reference scoring. Both methods use the same GPU, precision, model, prompts and sampling settings. Each detector scores the shared clean outputs separately.

32 task IDs calibrate the detection threshold; 32 different task IDs measure selection performance. This is exploratory work on a benchmark already studied in this project. An independent benchmark is still required for final validation.

Run cells in order. Results checkpoint to a separate Drive folder. Correctness remains pending until Docker evaluation on the laptop; this notebook never executes generated code.

In [1]:
USE_VSCODE = True
USE_DRIVE = True
TASK_COUNT = 64
CALIBRATION_TASKS = 32
SEEDS = [42]
PROFILE = 'pilot'  # Nine candidates. 'full' tests 17 candidates in a separate run.
SPLIT_SEED = 20261019
TARGET_FPR = 0.05  # Empirical calibration target, not a guaranteed population FPR.
MAX_QUALITY_LOSS = 0.05  # Five percentage points compared with shared clean code.

## Project and Drive

Select **Kernel > Colab > New Colab Server > GPU**. In VS Code Explorer, right-click `.tmp/codeguard-tuning-colab.zip` and choose **Upload to Colab**. Wait for the upload to finish before running this cell.

One writer per run. With `USE_DRIVE=False`, storage is temporary; download the export before disconnecting.

In [2]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
from IPython.display import Markdown, display

READY = False
if not Path('/content').is_dir():
    raise RuntimeError('Select a Google Colab GPU kernel in VS Code.')
from google.colab import drive, files
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Select a GPU Colab runtime, then rerun.')
DRIVE_ROOT = None
if USE_DRIVE:
    try:
        drive.mount('/content/drive')
    except Exception as exc:
        raise RuntimeError('In VS Code, run Colab: Mount Google Drive to Server, execute its cell and retry; '
                           'or set USE_DRIVE=False for temporary storage.') from exc
    DRIVE_ROOT = Path('/content/drive/MyDrive/CodeGuardLLM/colab-tuning')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

PROJECT = Path('/content/CodeGuardLLM-TUNING')
BUNDLE = Path('/content/codeguard-tuning-colab.zip')
saved_bundle = DRIVE_ROOT / BUNDLE.name if DRIVE_ROOT else None
if not BUNDLE.is_file() and saved_bundle and saved_bundle.is_file():
    shutil.copyfile(saved_bundle, BUNDLE)
if not BUNDLE.is_file():
    if USE_VSCODE:
        raise FileNotFoundError('Right-click .tmp/codeguard-tuning-colab.zip > Upload to Colab, then rerun.')
    print('Upload codeguard-tuning-colab.zip from your laptop.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Select only the project ZIP.')
    BUNDLE.write_bytes(next(iter(uploaded.values())))
    del uploaded
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        target = (PROJECT.parent / item.filename).resolve()
        if (not target.is_relative_to(PROJECT) or chr(92) in item.filename
                or (item.external_attr >> 16) & 0o170000 == 0o120000):
            raise ValueError('Invalid project archive path')
    if 'CodeGuardLLM-TUNING/src/codeguard/tuning.py' not in archive.namelist():
        raise ValueError('Upload the tuning project ZIP.')
    archive.extractall(PROJECT.parent)
if saved_bundle:
    shutil.copyfile(BUNDLE, saved_bundle)
SOURCE_SHA256 = hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))
from codeguard.cloud import Checkpoint
ENV = PROJECT / '.venv-colab'
PYTHON = str(ENV / 'bin/python')
CHECKPOINT = None
print('Project:', PROJECT)

Mounted at /content/drive
Project: /content/CodeGuardLLM-TUNING


In [3]:
from collections import deque
from datetime import datetime, timezone
import os, subprocess

LOG_DIR = PROJECT / '.tmp/notebook-logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def command(args, label='command'):
    log = LOG_DIR / (label + '-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f') + '.log')
    tail = deque(maxlen=30)
    print(f'Running {label}. Log: {log}', flush=True)
    with log.open('w', encoding='utf-8') as handle:
        with subprocess.Popen(
            list(map(str, args)), cwd=PROJECT, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
            bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1', 'MPLBACKEND': 'Agg'},
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    handle.write(line)
                    handle.flush()
                    tail.append(line)
                    if CHECKPOINT is not None:
                        CHECKPOINT.save()
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
            finally:
                if CHECKPOINT is not None:
                    CHECKPOINT.save(force=True)
    if status:
        detail = ''.join(tail)
        raise RuntimeError(f'{label} failed (exit {status}). Full log: {log}\n\n{detail}')

def run(*args, label='codeguard'):
    command([PYTHON, '-u', *args], label=label)

In [4]:
READY = False
if not Path(PYTHON).is_file():
    command([sys.executable, '-m', 'pip', 'install', 'virtualenv'], label='environment-tool')
    command([sys.executable, '-m', 'virtualenv', str(ENV)], label='environment')
command([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '--upgrade', 'pip'], label='pip')
run('-m', 'pip', 'install', 'torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126', label='pytorch')
run('-m', 'pip', 'install', '-e', '.[ml,data,analysis]', label='dependencies')
run('scripts/bootstrap_markllm.py', label='markllm')

run('-c', """
from copy import deepcopy
import gc, json
from pathlib import Path
import torch, yaml
from codeguard.backend import HFBackend
from codeguard.cloud import select_dtype
from codeguard.config import load_config, portable_config

assert torch.cuda.is_available(), 'Select a GPU Colab runtime'
base = load_config('configs/gpu.yaml')
base['model']['dtype'] = select_dtype(torch.cuda.get_device_capability()[0], torch.cuda.is_bf16_supported())
Path('.tmp/tuning-colab.yaml').write_text(yaml.safe_dump(base), encoding='utf-8')
metadata = {}
print('GPU:', torch.cuda.get_device_name(), '| Precision:', base['model']['dtype'])
for algorithm in ('KGW', 'SWEET'):
    config = deepcopy(base)
    config['watermark']['algorithm'] = algorithm
    if algorithm == 'SWEET':
        config['watermark']['entropy_threshold'] = 0.9
    config['generation']['max_new_tokens'] = 16
    backend = HFBackend(config)
    metadata[algorithm] = backend.metadata
    # Setup smoke tests only; these outputs never enter research results.
    for condition in ('clean', 'watermarked'):
        sample = backend.generate('Write a Python function add(a, b) returning a + b.', 'setup/tuning', 42, condition)
        score = backend.score(sample['raw_text'])
        assert score['status'] in ('ok', 'too_short', 'no_eligible_tokens')
        print('Setup:', algorithm, condition, sample['generated_tokens'], score['status'])
    del backend
    gc.collect()
    torch.cuda.empty_cache()
Path('.tmp/tuning-runtime.json').write_text(json.dumps({'backends': metadata, 'config': portable_config(base)}), encoding='utf-8')
""", label='check-both-methods')

CONFIG = PROJECT / '.tmp/tuning-colab.yaml'
runtime = json.loads((PROJECT / '.tmp/tuning-runtime.json').read_text())
settings = dict(tasks=TASK_COUNT, calibration_tasks=CALIBRATION_TASKS, seeds=SEEDS,
                profile=PROFILE, split_seed=SPLIT_SEED, target_fpr=TARGET_FPR, max_quality_loss=MAX_QUALITY_LOSS)
runtime.update(settings=settings, source_sha256=SOURCE_SHA256)
runtime['id'] = hashlib.sha256(json.dumps(runtime, sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR = PROJECT / 'runs/colab' / runtime['id'] / 'watermark_tuning'
backup = DRIVE_ROOT / runtime['id'] / 'results.zip' if DRIVE_ROOT else None
CHECKPOINT = Checkpoint(RUN_DIR, backup, PROJECT / '.tmp/checkpoints')
CHECKPOINT.restore()
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
run('scripts/tune_watermarks.py', 'prepare', '--config', CONFIG, '--output', RUN_DIR,
    '--tasks', TASK_COUNT, '--calibration-tasks', CALIBRATION_TASKS, '--profile', PROFILE,
    '--split-seed', SPLIT_SEED, '--target-fpr', TARGET_FPR, '--max-quality-loss', MAX_QUALITY_LOSS,
    '--seeds', *SEEDS, label='prepare-tuning')
CHECKPOINT.save(force=True)
READY = True
print('Results:', RUN_DIR)
print('Drive checkpoint:', backup if backup else 'disabled')

Running environment-tool. Log: /content/CodeGuardLLM-TUNING/.tmp/notebook-logs/environment-tool-20261019-123243-238064.log
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 30.7 MB/s eta 0:00:00
Running environment. Log: /content/CodeGuardLLM-TUNING/.tmp/notebook-logs/environment-20261019-123246-648987.log
created virtual environment CPython3.13.15.final.0-64-x86_64 in 472ms
  creator CPython3Posix(dest=/content/CodeGuardLLM-TUNING/.venv-colab, clear=False, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator
Running pip. Log: /content/CodeGuardLLM-TUNING/.tmp/notebook-logs/pip-20261019-123247-330355.log
Running pytorch. Log: /content/CodeGuardL

## Generate the matched pilot

Pilot: KGW gamma 0.1, 0.25, 0.5 at delta 2; SWEET entropy 0.3, 0.6, 0.9, 1.2 at delta 2, plus delta 1 and 3 at entropy 0.9. SWEET gamma remains 0.25. The full profile adds KGW delta 1/3 and the remaining SWEET entropy/strength combinations.

Sampling remains temperature 0.8, top-p 0.95, and at most 512 new tokens. Clean completions are generated once per task and seed. Existing complete records are skipped on resume. The same runtime and unchanged package/settings are required to restore the same Drive run. No completion-time estimate is assumed.

In [5]:
if not globals().get('READY', False):
    raise RuntimeError('Complete setup first.')
run('scripts/tune_watermarks.py', 'generate', '--output', RUN_DIR, label='matched-generation')

Running matched-generation. Log: /content/CodeGuardLLM-TUNING/.tmp/notebook-logs/matched-generation-20261019-123722-419524.log
Candidate: kgw-g0.1-d2
Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.
[1/128] HumanEval/58 clean: ok
[2/128] HumanEval/58 watermarked: ok
[3/128] HumanEval/50 clean: ok
[4/128] HumanEval/50 watermarked: ok
[5/128] HumanEval/101 clean: ok
[6/128] HumanEval/101 watermarked: ok
[7/128] HumanEval/67 clean: ok
[8/128] HumanEval/67 watermarked: ok
[9/128] HumanEval/120 clean: ok
[10/128] HumanEval/120 watermarked: ok
[11/128] HumanEval/133 clean: ok
[12/128] HumanEval/133 watermarked: ok
[13/128] HumanEval/30 clean: ok
[14/128] HumanEval/30 watermarked: ok
[15/128] HumanEval/19 clean: ok
[16/128] HumanEval/19 watermarked: ok
[17/128] HumanEval/22 clean: ok
[18/128] HumanEval/22 watermarked: ok
[19/128] HumanEval/55 clean: ok
[20/128] HumanEval/55 watermarked: ok
[21/128] HumanEval/47 clean: ok
[22/128] Huma

## Reference controls and calibration

Each distinct detector scores reference solutions; delta-only variants share reference scores. Calibration uses only clean and reference scores from calibration task IDs. Watermarked scores and selection tasks never determine the threshold.

The pilot targets an empirical 5% false-positive rate separately in each calibration control group. With 32 calibration tasks, tail estimates are coarse. HumanEval reference solutions are not an independent human-code corpus.

In [6]:
if not globals().get('READY', False):
    raise RuntimeError('Complete setup first.')
run('scripts/tune_watermarks.py', 'controls', '--output', RUN_DIR, label='reference-controls')

Running reference-controls. Log: /content/CodeGuardLLM-TUNING/.tmp/notebook-logs/reference-controls-20261019-131453-099133.log
[1/64] reference HumanEval/58: scored
[2/64] reference HumanEval/50: scored
[3/64] reference HumanEval/101: scored
[4/64] reference HumanEval/67: scored
[5/64] reference HumanEval/120: scored
[6/64] reference HumanEval/133: scored
[7/64] reference HumanEval/30: scored
[8/64] reference HumanEval/19: scored
[9/64] reference HumanEval/22: scored
[10/64] reference HumanEval/55: scored
[11/64] reference HumanEval/47: scored
[12/64] reference HumanEval/80: scored
[13/64] reference HumanEval/83: scored
[14/64] reference HumanEval/113: scored
[15/64] reference HumanEval/73: scored
[16/64] reference HumanEval/43: scored
[17/64] reference HumanEval/76: scored
[18/64] reference HumanEval/139: scored
[19/64] reference HumanEval/38: scored
[20/64] reference HumanEval/148: scored
[21/64] reference HumanEval/33: scored
[22/64] reference HumanEval/114: scored
[23/64] reference

In [7]:
run('scripts/tune_watermarks.py', 'analyze', '--output', RUN_DIR, label='tuning-analysis')
display(Markdown((RUN_DIR / 'analysis/report.md').read_text(encoding='utf-8')))

Running tuning-analysis. Log: /content/CodeGuardLLM-TUNING/.tmp/notebook-logs/tuning-analysis-20261019-131702-226215.log
Report: /content/CodeGuardLLM-TUNING/runs/colab/778e00ad4596cab7/watermark_tuning/analysis/report.md (correctness_pending)
Checkpoint saved: results.zip


# Matched KGW / SWEET tuning

Status: **correctness_pending**.

Exploratory tuning on previously studied HumanEval; unused tasks are not an independent benchmark.

Calibration uses 32 task IDs; selection uses 32 different task IDs. All seeds for a task stay in the same group. Each detector scores the same clean outputs.

Thresholds target an empirical FPR <= 5.0% separately on scorable clean and reference calibration code. This small pilot does not guarantee the population FPR. HumanEval references are not an independent human-code corpus. Unscored outputs remain explicit.

| Candidate | Z threshold | Selection clean FPR | Selection reference FPR | Detection / all | AUROC vs clean | Clean Pass@1 | WM Pass@1 | Correct and detected / all |
|---|---|---|---|---|---|---|---|---|
| kgw-g0.1-d2 | 2.429 | 0.000 | 0.000 | 0.062 | 0.605 | pending / unavailable | pending / unavailable | pending / unavailable |
| kgw-g0.25-d2 | 1.497 | 0.094 | 0.062 | 0.188 | 0.667 | pending / unavailable | pending / unavailable | pending / unavailable |
| kgw-g0.5-d2 | 1.627 | 0.062 | 0.094 | 0.219 | 0.709 | pending / unavailable | pending / unavailable | pending / unavailable |
| sweet-h0.3-d2 | 1.715 | 0.062 | 0.031 | 0.188 | 0.631 | pending / unavailable | pending / unavailable | pending / unavailable |
| sweet-h0.6-d2 | 1.549 | 0.094 | 0.031 | 0.156 | 0.631 | pending / unavailable | pending / unavailable | pending / unavailable |
| sweet-h0.9-d1 | 1.567 | 0.031 | 0.062 | 0.062 | 0.525 | pending / unavailable | pending / unavailable | pending / unavailable |
| sweet-h0.9-d2 | 1.567 | 0.031 | 0.062 | 0.125 | 0.578 | pending / unavailable | pending / unavailable | pending / unavailable |
| sweet-h0.9-d3 | 1.567 | 0.031 | 0.062 | 0.344 | 0.666 | pending / unavailable | pending / unavailable | pending / unavailable |
| sweet-h1.2-d2 | 1.281 | 0.125 | 0.156 | 0.156 | 0.538 | pending / unavailable | pending / unavailable | pending / unavailable |

Coverage, calibration counts, eligible-token counts and all metrics are in `metrics.csv`.

Repeated clean outputs and shared references are not additional independent observations. Native KGW and SWEET use different green-list seeding; this compares implementations.

Exploratory shortlist: unavailable; correctness pending or no candidate meets the quality budget.

Shortlisting maximizes joint correctness and detection within the quality-loss budget. Thresholds use calibration tasks only; selection metrics are tuning results. Confirm candidates with at least three seeds and evaluate frozen settings and thresholds on an independent benchmark. Do not call this pilot a final held-out result.


## Export, including interrupted work

Run the next cell after completion or an interrupted generation. In VS Code's **Colab > Contents** sidebar, refresh and download `/content/codeguard-tuning-results.zip`.

In [8]:
if 'RUN_DIR' not in globals() or not RUN_DIR.exists():
    raise RuntimeError('No tuning directory exists yet.')
CHECKPOINT.save(force=True)
EXPORT = shutil.make_archive(str(PROJECT.parent / 'codeguard-tuning-results'), 'zip', RUN_DIR.parent, RUN_DIR.name)
print('Results archive:', EXPORT)
if USE_VSCODE:
    print('Colab > Contents > refresh > codeguard-tuning-results.zip > Download...')
else:
    files.download(EXPORT)

Checkpoint saved: results.zip
Results archive: /content/codeguard-tuning-results.zip
Colab > Contents > refresh > codeguard-tuning-results.zip > Download...


## Finish correctness evaluation on the laptop

Start Docker Desktop's Linux engine. Run these commands in the **local VS Code PowerShell terminal**, from the project directory. Extract to a new folder; adjust the downloaded filename if needed.

```powershell
Expand-Archive -LiteralPath "$env:USERPROFILE\Downloads\codeguard-tuning-results.zip" -DestinationPath runs/tuning-import
.\.venv\Scripts\python.exe scripts/tune_watermarks.py evaluate --output runs/tuning-import/watermark_tuning
.\.venv\Scripts\python.exe scripts/tune_watermarks.py analyze --output runs/tuning-import/watermark_tuning
```

Open `runs/tuning-import/watermark_tuning/analysis/report.md`. These local steps use saved scores and task snapshots; no model loading or GPU rescoring is needed. The exploratory shortlist appears only when selection correctness is complete, and keeps the five-percentage-point correctness-loss budget.

The shortlist is a development result. Confirm promising settings with at least three seeds, then freeze settings and calibration thresholds before using an independent evaluation benchmark. The remaining HumanEval tasks have already been studied in earlier project runs and should not be presented as an independent final test.